# First overfit preparation: Trackastra + spatial cache

This notebook assumes the Stage 1, Stage 2, and CC-only Stage 3 artifacts have already
been saved under `data/learned/stirnet/first_overfit/BlastoSPIM1_F22_030_034/`.

It does two things only:

1. runs Trackastra pass 1 on the five prepared frames and saves the graph/results;
2. prepares and saves the target-frame spatial channels needed by STIR-Net, including
   the per-instance physical EDT.

The actual STIR-Net forward/overfit experiment is intentionally kept in the next notebook.


In [ ]:
from pathlib import Path
import json
import pickle
import gc

import numpy as np
import torch


def find_repo_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()

    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "learned").exists():
            return path

    raise RuntimeError("Could not locate repository root.")


PROJECT_ROOT = find_repo_root()

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

print("Project root:", PROJECT_ROOT)
print("Data dir    :", DATA_DIR)
print("Exists      :", DATA_DIR.exists())

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

In [ ]:
frame_numbers = np.load(
    DATA_DIR / "frame_numbers.npy",
)

raw_movie = np.load(
    DATA_DIR / "raw_movie.npy",
    mmap_mode="r",
)

processed_movie = np.load(
    DATA_DIR / "processed_movie.npy",
    mmap_mode="r",
)

binary_movie = np.load(
    DATA_DIR / "binary_movie.npy",
    mmap_mode="r",
)

instance_movie = np.load(
    DATA_DIR / "instance_movie.npy",
    mmap_mode="r",
)

markers_movie = np.load(
    DATA_DIR / "markers_movie.npy",
    mmap_mode="r",
)

gt_movie = np.load(
    DATA_DIR / "gt_movie.npy",
    mmap_mode="r",
)

with open(
    DATA_DIR / "metadata.json",
    "r",
    encoding="utf-8",
) as f:
    metadata = json.load(f)


print("Frames    :", frame_numbers.tolist())
print("Raw       :", raw_movie.shape, raw_movie.dtype)
print("Processed :", processed_movie.shape, processed_movie.dtype)
print("Binary    :", binary_movie.shape, binary_movie.dtype)
print("Instances :", instance_movie.shape, instance_movie.dtype)
print("Markers   :", markers_movie.shape, markers_movie.dtype)
print("GT        :", gt_movie.shape, gt_movie.dtype)

SPACING_ZYX_UM = tuple(
    metadata["spacing_zyx_um"]
)

print()
print("Spacing ZYX:", SPACING_ZYX_UM, "µm")

assert raw_movie.shape == instance_movie.shape == gt_movie.shape
assert raw_movie.ndim == 4
assert len(frame_numbers) == raw_movie.shape[0]

In [ ]:
TARGET_LOCAL_T = 2
TARGET_FRAME = int(frame_numbers[TARGET_LOCAL_T])

TIME_OFFSETS = np.arange(
    len(frame_numbers)
) - TARGET_LOCAL_T


print("Temporal window")

for local_t, (frame, offset) in enumerate(
    zip(frame_numbers, TIME_OFFSETS)
):
    marker = "<-- TARGET" if local_t == TARGET_LOCAL_T else ""

    print(
        f"local_t={local_t} | "
        f"F22_{int(frame):03d} | "
        f"offset={int(offset):+d} "
        f"{marker}"
    )


print()
print("Target current instances:")

target_instance_ids = np.unique(
    instance_movie[TARGET_LOCAL_T]
)

target_instance_ids = target_instance_ids[
    target_instance_ids > 0
]

target_gt_ids = np.unique(
    gt_movie[TARGET_LOCAL_T]
)

target_gt_ids = target_gt_ids[
    target_gt_ids > 0
]

print("Current instances:", len(target_instance_ids))
print("GT instances     :", len(target_gt_ids))

In [ ]:
from trackastra.model import Trackastra


# Keep the complete spatial volume.
# np.asarray on the memory-mapped arrays should avoid an unnecessary
# explicit copy here.
imgs = np.asarray(raw_movie)
masks = np.asarray(instance_movie)

assert imgs.ndim == 4
assert masks.ndim == 4
assert imgs.shape == masks.shape
assert imgs.shape[0] == 5

print("Trackastra input")
print("-----------------")
print("Images:", imgs.shape, imgs.dtype)
print("Masks :", masks.shape, masks.dtype)

print()
print("Instance counts:")

for local_t, frame in enumerate(frame_numbers):
    ids = np.unique(masks[local_t])
    count = int(np.count_nonzero(ids > 0))

    marker = " <-- TARGET" if local_t == TARGET_LOCAL_T else ""

    print(
        f"F22_{int(frame):03d}: "
        f"{count:3d} instances"
        f"{marker}"
    )

In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    print(
        "CUDA allocated:",
        f"{torch.cuda.memory_allocated() / 1024**2:.1f} MB",
    )
    print(
        "CUDA reserved :",
        f"{torch.cuda.memory_reserved() / 1024**2:.1f} MB",
    )

print()
print(
    "Raw array size :",
    f"{imgs.nbytes / 1024**3:.2f} GB",
)

print(
    "Mask array size:",
    f"{masks.nbytes / 1024**3:.2f} GB",
)

In [ ]:
trackastra = Trackastra.from_pretrained(
    "ctc",
    device="cuda",
)

print("Trackastra loaded.")
print("Model type:", type(trackastra))

if torch.cuda.is_available():
    print(
        "CUDA allocated:",
        f"{torch.cuda.memory_allocated() / 1024**2:.1f} MB",
    )

In [ ]:
track_graph, masks_tracked = trackastra.track(
    imgs=imgs,
    masks=masks,
    mode="greedy",
    n_workers=0,
    batch_size=4,
)

print()
print("Trackastra result")
print("-----------------")

print(
    "Tracked masks:",
    masks_tracked.shape,
    masks_tracked.dtype,
)

print(
    "Graph nodes:",
    track_graph.number_of_nodes(),
)

print(
    "Graph edges:",
    track_graph.number_of_edges(),
)

if torch.cuda.is_available():
    print()
    print(
        "Peak GPU allocation:",
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB",
    )

In [ ]:
print("Trackastra nodes by frame")
print("-------------------------")

total_nodes = 0

for local_t, frame in enumerate(frame_numbers):

    node_count = sum(
        int(data["time"]) == local_t
        for _, data in track_graph.nodes(data=True)
    )

    total_nodes += node_count

    input_count = int(
        np.count_nonzero(
            np.unique(masks[local_t]) > 0
        )
    )

    marker = " <-- TARGET" if local_t == TARGET_LOCAL_T else ""

    print(
        f"F22_{int(frame):03d} | "
        f"input instances={input_count:3d} | "
        f"graph nodes={node_count:3d}"
        f"{marker}"
    )

print()
print("Counted graph nodes:", total_nodes)
print(
    "Graph node total   :",
    track_graph.number_of_nodes(),
)

assert total_nodes == track_graph.number_of_nodes()

In [ ]:
print("First 5 nodes")
print("-------------")

for node_id, data in list(
    track_graph.nodes(data=True)
)[:5]:
    print(
        "node:",
        node_id,
        "|",
        data,
    )


print()
print("First 5 edges")
print("-------------")

for src, dst, data in list(
    track_graph.edges(data=True)
)[:5]:
    print(
        src,
        "->",
        dst,
        "|",
        data,
    )

In [ ]:
TRACKASTRA_DIR = DATA_DIR / "trackastra"

TRACKASTRA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

np.save(
    TRACKASTRA_DIR / "masks_tracked.npy",
    masks_tracked,
)

with open(
    TRACKASTRA_DIR / "track_graph.pkl",
    "wb",
) as f:
    pickle.dump(
        track_graph,
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


print("Saved:")
print(
    TRACKASTRA_DIR / "masks_tracked.npy"
)
print(
    TRACKASTRA_DIR / "track_graph.pkl"
)

In [ ]:
# Trackastra is now cached on disk. Release its GPU state before the CPU-side
# spatial-cache preparation below.

if "trackastra" in globals():
    del trackastra

if "masks_tracked" in globals():
    del masks_tracked

if "imgs" in globals():
    del imgs

if "masks" in globals():
    del masks

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    print(
        "CUDA allocated after releasing Trackastra:",
        f"{torch.cuda.memory_allocated() / 1024**2:.1f} MB",
    )


## Cache target-frame STIR-Net spatial inputs

The repository's current `build_spatial_channels()` computes an EDT over the full volume
once per instance. For this large volume that is unnecessarily expensive. The helper below
computes the same per-instance EDT inside each instance's local bounding box and writes the
resulting target-frame channels to disk.

EDT values are normalized by `d_ref`, matching the STIR-Net data contract.


In [ ]:
from scipy import ndimage as ndi

from learned.stirnet.data.sample_builder import robust_normalize
from learned.stirnet.data.targets import (
    estimate_dref_um,
    make_instance_boundary,
)


STIRNET_SOURCE_DIR = DATA_DIR / "stirnet_source"
STIRNET_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

target_raw = np.asarray(raw_movie[TARGET_LOCAL_T])
target_instances = np.asarray(instance_movie[TARGET_LOCAL_T])
target_gt = np.asarray(gt_movie[TARGET_LOCAL_T])
target_markers = np.asarray(markers_movie[TARGET_LOCAL_T])

dref_um = estimate_dref_um(
    target_gt,
    SPACING_ZYX_UM,
)

print("Target frame:", TARGET_FRAME)
print("Shape       :", target_raw.shape)
print("d_ref       :", f"{dref_um:.4f} µm")


In [ ]:
def per_instance_edt_local(
    labels: np.ndarray,
    spacing_um,
    dref_um: float,
) -> np.ndarray:
    """Per-instance physical EDT using local bounding boxes."""

    labels = np.asarray(labels)
    spacing_um = tuple(float(x) for x in spacing_um)

    edt = np.zeros(
        labels.shape,
        dtype=np.float32,
    )

    objects = ndi.find_objects(labels)

    for label_id, bbox in enumerate(objects, start=1):
        if bbox is None:
            continue

        expanded = []

        for axis_slice, axis_size in zip(bbox, labels.shape):
            expanded.append(
                slice(
                    max(0, axis_slice.start - 1),
                    min(axis_size, axis_slice.stop + 1),
                )
            )

        expanded = tuple(expanded)

        local_mask = (
            labels[expanded] == label_id
        )

        if not local_mask.any():
            continue

        # Explicit zero padding guarantees background outside a component
        # even when the component touches the acquisition boundary.
        padded = np.pad(
            local_mask,
            pad_width=1,
            mode="constant",
            constant_values=False,
        )

        local_edt = ndi.distance_transform_edt(
            padded,
            sampling=spacing_um,
        )[1:-1, 1:-1, 1:-1]

        local_edt = (
            local_edt
            / max(float(dref_um), 1e-6)
        ).astype(np.float32, copy=False)

        out_view = edt[expanded]
        out_view[local_mask] = local_edt[local_mask]

    return edt


EDT_PATH = STIRNET_SOURCE_DIR / "edt_target.npy"

if EDT_PATH.exists():
    print("EDT cache already exists; reusing:")
    print(EDT_PATH)
    edt_target = np.load(
        EDT_PATH,
        mmap_mode="r",
    )
else:
    print("Computing target-frame per-instance EDT...")
    edt_target = per_instance_edt_local(
        target_instances,
        SPACING_ZYX_UM,
        dref_um,
    )
    np.save(
        EDT_PATH,
        edt_target,
    )
    print("Saved:", EDT_PATH)

print(
    "EDT:",
    edt_target.shape,
    edt_target.dtype,
    "max=",
    float(np.max(edt_target)),
)


In [ ]:
RAW_NORM_PATH = STIRNET_SOURCE_DIR / "raw_norm_target.npy"
FOREGROUND_PATH = STIRNET_SOURCE_DIR / "foreground_target.npy"
BOUNDARY_PATH = STIRNET_SOURCE_DIR / "boundary_target.npy"
MARKER_PATH = STIRNET_SOURCE_DIR / "marker_heatmap_target.npy"
DREF_PATH = STIRNET_SOURCE_DIR / "dref_um.npy"

if not RAW_NORM_PATH.exists():
    np.save(
        RAW_NORM_PATH,
        robust_normalize(target_raw),
    )

if not FOREGROUND_PATH.exists():
    np.save(
        FOREGROUND_PATH,
        (target_instances > 0),
    )

if not BOUNDARY_PATH.exists():
    np.save(
        BOUNDARY_PATH,
        make_instance_boundary(target_instances),
    )

if not MARKER_PATH.exists():
    np.save(
        MARKER_PATH,
        (target_markers > 0).astype(np.float32),
    )

np.save(
    DREF_PATH,
    np.asarray(dref_um, dtype=np.float32),
)

source_metadata = {
    "target_local_t": int(TARGET_LOCAL_T),
    "target_frame": int(TARGET_FRAME),
    "spacing_zyx_um": [float(x) for x in SPACING_ZYX_UM],
    "dref_um": float(dref_um),
    "channels": {
        "raw_norm": RAW_NORM_PATH.name,
        "foreground": FOREGROUND_PATH.name,
        "edt_normalized_by_dref": EDT_PATH.name,
        "boundary": BOUNDARY_PATH.name,
        "marker_heatmap": MARKER_PATH.name,
    },
}

with open(
    STIRNET_SOURCE_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        source_metadata,
        f,
        indent=2,
    )

print("Saved STIR-Net source cache:")
for p in sorted(STIRNET_SOURCE_DIR.iterdir()):
    print(
        f"  {p.name:28s}"
        f"{p.stat().st_size / 1024**2:10.2f} MB"
    )


## Preparation complete

After this notebook succeeds, the overfit notebook can restart from disk. It does not need
to rerun Trackastra or recompute the target-frame EDT.


In [ ]:
required_files = [
    DATA_DIR / "frame_numbers.npy",
    DATA_DIR / "raw_movie.npy",
        DATA_DIR / "processed_movie.npy",
        DATA_DIR / "binary_movie.npy",
    DATA_DIR / "instance_movie.npy",
    DATA_DIR / "markers_movie.npy",
    DATA_DIR / "gt_movie.npy",
    DATA_DIR / "metadata.json",
    TRACKASTRA_DIR / "track_graph.pkl",
    TRACKASTRA_DIR / "masks_tracked.npy",
    STIRNET_SOURCE_DIR / "raw_norm_target.npy",
    STIRNET_SOURCE_DIR / "foreground_target.npy",
    STIRNET_SOURCE_DIR / "edt_target.npy",
    STIRNET_SOURCE_DIR / "boundary_target.npy",
    STIRNET_SOURCE_DIR / "marker_heatmap_target.npy",
    STIRNET_SOURCE_DIR / "dref_um.npy",
]

missing = [
    p
    for p in required_files
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing preparation artifacts:\n"
        + "\n".join(str(p) for p in missing)
    )

print("Preparation cache is complete.")
print("Ready for the STIR-Net overfit notebook.")
